# Batch Norm Behaves Differently When You Are Watching

**Topic:** Batch normalization &nbsp;·&nbsp; **Difficulty:** Hard

Batch norm is the classic layer whose output depends on the *other examples in the
batch*. In training it normalizes with the batch's own statistics, and at inference
it uses a running average accumulated during training. Those are different
numbers, and if your running statistics are stale a model that scored 94% in
training can score 70% in production (illustrative numbers, not a measurement).

Implement both paths and the running-statistics update, then measure how far
apart they are.

## The data (created in the next cell)

- `batches`: 50 batches of shape `(64, 32)`, drawn from a distribution whose
  mean **drifts** over the course of training.
- `momentum = 0.1` for the running statistics, `eps = 1e-5`.
- `gamma = 1`, `beta = 0` (so the only effect is the normalization itself, and you
  do not need to code these two constants).

## Your task

With **NumPy only**:

1. Initialize `running_mean = zeros(32)`, `running_var = ones(32)`.
2. For each batch in order:
   - compute the batch mean and **biased** variance (`ddof=0`) per feature,
   - the training output is `(x - batch_mean) / sqrt(batch_var + eps)`,
   - update `running_mean = (1 - momentum) * running_mean + momentum * batch_mean`
     and the same for `running_var` using the **unbiased** batch variance
     (`ddof=1`), which is what PyTorch does.
3. After the loop, evaluate the **final** batch in inference mode, using the
   running statistics instead of the batch's own.
4. Set `answers` to a dict with `"running_mean"` `(32,)`, `"running_var"` `(32,)`,
   `"train_out_mean"` (mean of the last batch's training-mode output),
   `"train_out_std"` (its std), `"eval_out_mean"`, `"eval_out_std"`, and
   `"mode_gap"`, the mean absolute difference between the two outputs.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the training-mode output is standardized (mean 0, std 1) while the
eval-mode output's mean is pushed off zero, because the running statistics lag
the drift. Only the eval mean is gated, since its standard deviation is not
compared with 1.

In [ ]:
import numpy as np

rng = np.random.default_rng(2525)
n_batches, bs, d = 50, 64, 32
batches = []
for i in range(n_batches):
    drift = 0.06 * i
    batches.append(rng.normal(drift, 1.0 + 0.02 * i, (bs, d)))
momentum, eps = 0.1, 1e-5
print(f"{n_batches} batches of {bs}x{d}, with the feature mean drifting from 0 to "
      f"{0.06 * (n_batches - 1):.2f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("running_mean", [32], 76.70231080858312, -1.5150318758460721, 1e-06),
    ("running_var", [32], 104.1718108468894, -1.7687711794677021, 1e-06),
    ("train_out_mean", [], 1.3877787807814457e-17, -2.323669240557313e-18, 1e-06),
    ("train_out_std", [], 0.9999985907480177, -0.16743777885214486, 1e-06),
    ("eval_out_mean", [], 0.28798121315128755, -0.048219002633922724, 1e-06),
    ("eval_out_std", [], 1.0622959032809576, -0.17786871719093875, 1e-06),
    ("mode_gap", [], 0.28949188440881185, -0.048471946430319914, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "mode_gap" in answers:
    _t = abs(answers["train_out_mean"]) < 1e-9 and abs(answers["train_out_std"] - 1) < 1e-3
    _e = abs(answers["eval_out_mean"]) > 0.05
    print(f"  {'train mode standardized':<24} {'ok' if _t else 'wrong'}")
    print(f"  {'eval mode lags drift':<24} {'ok' if _e else 'wrong'}  "
          f"(eval mean {answers['eval_out_mean']:+.4f})")
    _ok = _ok and _t and _e

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The running average is an exponential moving average with `momentum = 0.1`, so it
reflects roughly the last ten batches. When the data drifts, it is always
chasing and never arrives. The training-mode output is standardized by
construction (mean exactly 0, and std 1 up to the eps term, per feature). The
eval-mode output is not, and the
gap is the drift the running statistics have not caught up with. Note also the
biased/unbiased asymmetry: the normalization uses `ddof=0` but the running
variance stores `ddof=1`, which is a real detail of the reference
implementation.

```python
import numpy as np

running_mean = np.zeros(d)
running_var = np.ones(d)

for x in batches:
    bm = x.mean(axis=0)
    bv = x.var(axis=0)                 # biased, used for normalizing
    _ = (x - bm) / np.sqrt(bv + eps)
    bv_unbiased = x.var(axis=0, ddof=1)
    running_mean = (1 - momentum) * running_mean + momentum * bm
    running_var = (1 - momentum) * running_var + momentum * bv_unbiased

last = batches[-1]
bm = last.mean(axis=0)
bv = last.var(axis=0)
train_out = (last - bm) / np.sqrt(bv + eps)
eval_out = (last - running_mean) / np.sqrt(running_var + eps)

answers = {
    "running_mean": running_mean,
    "running_var": running_var,
    "train_out_mean": float(train_out.mean()),
    "train_out_std": float(train_out.std()),
    "eval_out_mean": float(eval_out.mean()),
    "eval_out_std": float(eval_out.std()),
    "mode_gap": float(np.abs(train_out - eval_out).mean()),
}
```